# Lesson 7: Time Series Forecasting: ARIMA & SVR

**Track:** AI Engineer — Phase 1: Data Foundations & Classical ML  
**Runtime:** Google Colab Free Tier (CPU)  
**Sources:** `.ref/ML-For-Beginners/ML-For-Beginners-main/7-TimeSeries/` (1-Introduction, 2-ARIMA, 3-SVR)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR


## 1. Chronological Split & SARIMAX Forecast


In [ ]:
rng = np.random.default_rng(42)
t = np.arange(120)
load = 100 + 0.4 * t + 12 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 2, 120)
train, test = load[:96], load[96:]

sarimax = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
arima_pred = sarimax.forecast(steps=len(test))
arima_mape = np.mean(np.abs((test - arima_pred) / test)) * 100
print(f'SARIMAX Test MAPE: {arima_mape:.2f}%')


## 2. Lagged Window Framing + SVR


In [ ]:
def make_lags(series, window=12):
    X, y = [], []
    for i in range(window, len(series)):
        X.append(series[i - window:i])
        y.append(series[i])
    return np.array(X), np.array(y)

X_all, y_all = make_lags(load, window=12)
split = 96 - 12
svr = make_pipeline(MinMaxScaler(), SVR(kernel='rbf', C=50.0)).fit(X_all[:split], y_all[:split])
svr_pred = svr.predict(X_all[split:])
svr_mape = np.mean(np.abs((y_all[split:] - svr_pred) / y_all[split:])) * 100
print(f'Lagged SVR Test MAPE: {svr_mape:.2f}%')


## Exercise
Plot `test`, `arima_pred`, and `svr_pred` on the same chart to compare seasonal tracking.
